# This code is used to create reference list what food contain what nutrients

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import json

# Setting paths
base_path = 'nutritionverse-data'
images_path = os.path.join(base_path, 'nutritionverse-manual/nutritionverse-manual/images')
metadata_path = os.path.join(base_path, 'nutritionverse_dish_metadata3.csv')
coco_json_path = os.path.join(base_path, 'nutritionverse-manual/nutritionverse-manual/images/_annotations.coco.json')
splits_path = os.path.join(base_path, 'nutritionverse-manual/nutritionverse-manual/updated-manual-dataset-splits.csv')

# Load COCO format image metadata
with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

print(f"COCO metadata loaded with {len(coco_data['images'])} images and {len(coco_data['annotations'])} annotations")

# Optionally extract image information from COCO format
coco_images = pd.DataFrame(coco_data['images'])
print("COCO images structure:")
print(coco_images.columns.tolist())

# Create a mapping from image_id to file_name
image_id_to_filename = {img['id']: img['file_name'] for img in coco_data['images']}
print(f"Created mapping for {len(image_id_to_filename)} image IDs to filenames")

# Load dataset splits if available
if os.path.exists(splits_path):
    splits_df = pd.read_csv(splits_path)
    print(f"Dataset splits loaded with {len(splits_df)} entries")
    print(f"Split distribution: {splits_df['category'].value_counts().to_dict()}")
else:
    print("Dataset splits file not found. Will use random splitting.")
    splits_df = None

# Load metadata
metadata_df = pd.read_csv(metadata_path)

# Display first few rows of metadata
print("Dataset Metadata Overview:")
print(f"Total entries: {len(metadata_df)}")
metadata_df.head()


COCO metadata loaded with 905 images and 4392 annotations
COCO images structure:
['id', 'license', 'file_name', 'height', 'width', 'date_captured']
Created mapping for 905 image IDs to filenames
Dataset splits loaded with 889 entries
Split distribution: {'Train': 624, 'Val': 265}
Dataset Metadata Overview:
Total entries: 251


Index(['dish_id', 'total_food_weight', 'total_calories', 'total_fats',
       'total_carbohydrates', 'total_protein', 'total_calcium', 'total_iron',
       'total_magnesium', 'total_potassium',
       ...
       'fat(g)_7', 'carbohydrates(g)_7', 'protein(g)_7', 'calcium(mg)_7',
       'iron(mg)_7', 'magnesium(mg)_7', 'potassium(mg)_7', 'sodium(mg)_7',
       'vitamin_d(µg)_7', 'vitamin_b12(µg)_7'],
      dtype='object', length=104)

In [16]:
food_nutrient = {}
for index, item in metadata_df.iterrows():
    for i in range(1, 8):
        food_type_col = f'food_item_type_{i}'
        
        # Skip if this food item doesn't exist
        if food_type_col not in item or pd.isna(item[food_type_col]):
            continue
        
        food_type = item[food_type_col]
        
        # Initialize dictionary entry if this food item type doesn't exist yet
        if food_type not in food_nutrient:
            food_nutrient[food_type] = {
                'calories': 0,
                'fat': 0,
                'carbohydrates': 0,
                'protein': 0,
                'calcium': 0,
                'iron': 0,
                'magnesium': 0,
                'potassium': 0,
                'sodium': 0,
                'vitamin_d': 0,
                'vitamin_b12': 0
            }
        
        # Add nutrients for this instance only if the values are not NaN or null
        if pd.notna(item[f'calories(kCal)_{i}']):
            food_nutrient[food_type]['calories'] = item[f'calories(kCal)_{i}']
        if pd.notna(item[f'fat(g)_{i}']):
            food_nutrient[food_type]['fat'] = item[f'fat(g)_{i}']
        if pd.notna(item[f'carbohydrates(g)_{i}']):
            food_nutrient[food_type]['carbohydrates'] = item[f'carbohydrates(g)_{i}']
        if pd.notna(item[f'protein(g)_{i}']):
            food_nutrient[food_type]['protein'] = item[f'protein(g)_{i}']
        if pd.notna(item[f'calcium(mg)_{i}']):
            food_nutrient[food_type]['calcium'] = item[f'calcium(mg)_{i}']
        if pd.notna(item[f'iron(mg)_{i}']):
            food_nutrient[food_type]['iron'] = item[f'iron(mg)_{i}']
        if pd.notna(item[f'magnesium(mg)_{i}']):
            food_nutrient[food_type]['magnesium'] = item[f'magnesium(mg)_{i}']
        if pd.notna(item[f'potassium(mg)_{i}']):
            food_nutrient[food_type]['potassium'] = item[f'potassium(mg)_{i}']
        if pd.notna(item[f'sodium(mg)_{i}']):
            food_nutrient[food_type]['sodium'] = item[f'sodium(mg)_{i}']
        if pd.notna(item[f'vitamin_d(µg)_{i}']):
            food_nutrient[food_type]['vitamin_d'] = item[f'vitamin_d(µg)_{i}']
        if pd.notna(item[f'vitamin_b12(µg)_{i}']):
            food_nutrient[food_type]['vitamin_b12'] = item[f'vitamin_b12(µg)_{i}']

In [20]:
# Convert the food_nutrient dictionary to a DataFrame
food_nutrient_df = pd.DataFrame.from_dict(food_nutrient, orient='index').reset_index()

# Rename the 'index' column to 'food_type'
food_nutrient_df.rename(columns={'index': 'food_type'}, inplace=True)

# Display the resulting DataFrame
food_nutrient_df.head()
food_nutrient_df.to_csv('food_nutrients.csv')